# Analisi Consumo di Suolo in Italia

**Fonte:** ISPRA — Consumo di suolo, dinamiche territoriali e servizi ecosistemici  
**Dati:** DataCivicLab (clean parquet da GCS)  
**Copertura:** 2006–2024  
**Livello:** Comunale, regionale, nazionale

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

In [ ]:
df = pd.read_parquet(
    'https://storage.googleapis.com/dataciviclab-clean/ispra_consumo_suolo/2024/ispra_consumo_suolo_2024_clean.parquet'
)
df.head(3)

## 1. Pulizia e check preliminari

In [ ]:
df.info()

In [ ]:
# Valori mancanti
df.isnull().sum()

In [ ]:
# Duplicati
print('Duplicati:', df.duplicated().sum())

In [ ]:
# Statistiche descrittive
df.describe()

In [ ]:
# Regioni presenti
df['regione'].value_counts()

## 2. Trend nazionale annuale

In [ ]:
# Solo righe annuali (escludo periodi pluriennali 2006-2012, 2012-2015)
df_annuale = df[df['periodo'].str.contains(r'^\\d{4}-\\d{4}$', na=False)].copy()

trend_nazionale = df_annuale.groupby('anno').agg(
    tot_inc_netto_ha=('incremento_netto_ha', 'sum'),
    tot_inc_lordo_ha=('incremento_lordo_ha', 'sum'),
    tot_ripristino_ha=('ripristino_ha', 'sum'),
    media_stock_pct=('stock_pct', 'mean')
).reset_index()

trend_nazionale

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(trend_nazionale['anno'], trend_nazionale['tot_inc_netto_ha'], 
           marker='o', linewidth=2, color='#d32f2f')
ax[0].set_title('Incremento netto consumo suolo (ha/anno)', fontsize=13)
ax[0].set_xlabel('Anno')
ax[0].set_ylabel('Ettari')
ax[0].grid(True, alpha=0.3)

ax[1].plot(trend_nazionale['anno'], trend_nazionale['media_stock_pct'],
           marker='s', linewidth=2, color='#1976d2')
ax[1].set_title('Stock medio consumo suolo (% per comune)', fontsize=13)
ax[1].set_xlabel('Anno')
ax[1].set_ylabel('% medio')
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('trend_nazionale.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Confronto regionale (2024)

In [ ]:
df_2024 = df[df['anno'] == 2024]

regioni_2024 = df_2024.groupby('regione').agg(
    tot_inc_netto_ha=('incremento_netto_ha', 'sum'),
    tot_stock_ha=('stock_ha', 'sum'),
    avg_stock_pct=('stock_pct', 'mean'),
    n_comuni=('comune', 'nunique')
).reset_index().sort_values('tot_inc_netto_ha', ascending=False)

regioni_2024.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
top10 = regioni_2024.head(10)
bars = ax.barh(top10['regione'], top10['tot_inc_netto_ha'], color='#d32f2f')
ax.set_xlabel('Ettari consumati nel 2024')
ax.set_title('Top 10 regioni per nuovo consumo suolo (2024)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('regioni_2024.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Province con più consumo pro-capite

In [ ]:
province_2024 = df_2024.groupby(['regione', 'provincia']).agg(
    tot_inc_netto_ha=('incremento_netto_ha', 'sum'),
    avg_stock_pct=('stock_pct', 'mean')
).reset_index().sort_values('avg_stock_pct', ascending=False)

province_2024.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
top_prov = province_2024.head(15)
colors = ['#d32f2f' if r == 'Lombardia' else '#1976d2' for r in top_prov['regione']]
bars = ax.barh(top_prov['provincia'] + ' (' + top_prov['regione'] + ')',
               top_prov['avg_stock_pct'], color=colors)
ax.set_xlabel('% medio suolo consumato')
ax.set_title('Top 15 province per % consumo suolo (2024)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('province_2024.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Insight chiave

1. **Trend**: Il consumo di suolo in Italia [inserire trend osservato: cala/aumenta/stabile]
2. **Regione**: La regione con più nuovo consumo nel 2024 è [nome] con [valore] ettari
3. **Provincia**: La provincia con la più alta percentuale di suolo consumato è [nome] con [valore]%
4. **Nord vs Sud**: Le regioni del Nord consumano [più/meno] suolo pro-capite rispetto al Sud

*(Compila dopo aver visto i numeri)*

In [ ]:
print('Analisi completata.')